# TP06: training better networks

**MSA-DATB09-03 · Neural Networks & Deep Learning Foundations · Session 6 / 8** · Guillaume Saint-Cirgue

*Estimated time: 85 minutes*

In the course we saw that a big network trained on few images **memorises** them: the training loss goes
to 0 while the validation loss rises. We saw the tools to fight this (weight decay, dropout, early
stopping), batch normalisation, better optimisers (momentum, Adam, schedules) and how to read a sick
training run. In this notebook you provoke overfitting on FashionMNIST, then you write each tool yourself
and measure what it changes. You finish as a "training doctor": three broken runs to diagnose and fix.

**How it works.** Each exercise is a cell with a zone to complete:

```python
### START CODE HERE ###

### END CODE HERE ###
```

Write your code between the two markers, then run the cell: the `assert` lines below the zone check your
answer and print `Exercise validated!` when it is correct. If an assert fails, read its message: it tells
you what to look at.

**Run it** on Google Colab (File > Upload notebook) or locally with the course `requirements.txt`. A CPU is
enough: the whole notebook trains small networks on 2 000 images (a few minutes in total).

## Imports

- `numpy`: small computations by hand.
- `pandas`: the comparison table of Part B.
- `matplotlib`: training curves.
- `torch`, `torch.nn`: the networks; `DataLoader` and `TensorDataset` serve mini-batches from tensors.
- `torchvision.datasets`: downloads FashionMNIST.
- `copy`: `copy.deepcopy` makes an independent copy of the weights (for early stopping).

In [ ]:
import os
import copy
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

FAST_RUN = os.environ.get("FAST_RUN", "0") == "1"   # True: shorter trainings for a quick check
DATA_DIR = os.environ.get("DATA_DIR", "./data")
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(2)

EPOCHS = 15 if FAST_RUN else 30        # Parts A and B (we want to see overfitting)
EPOCHS_SHORT = 6 if FAST_RUN else 10   # Parts C, D, E
print("FAST_RUN:", FAST_RUN)

## Helper functions (run this cell, no need to modify it)

This cell prepares the data and gives you the training tools of sessions 4 and 5:

- **Data**: 2 000 training images and 5 000 validation images taken from the FashionMNIST training set
  (disjoint), flattened to vectors of 784 pixels and standardised with the mean and standard deviation of the
  training images. 2 000 images is deliberately small: we want to overfit.
- `train_one_epoch(model, loader, loss_fn, optimizer)`: one pass over the training data with the 5 lines of
  the training loop; returns the mean training loss.
- `evaluate(model, loader, loss_fn)`: returns `(mean loss, accuracy)` in evaluation mode, without gradients.
- `fit(model, optimizer, n_epochs, ...)`: trains for `n_epochs` and returns a **history**, a dictionary of
  lists with one value per epoch: `"train_loss"`, `"train_acc"`, `"val_loss"`, `"val_acc"`, `"lr"`.
  The train metrics are measured in evaluation mode on the whole training set after each epoch, so that
  train and validation are measured the same way.
- `plot_history(history)` and `plot_compare(histories, key)`: plots.

In [ ]:
full = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
X_all = full.data.float().div(255).reshape(-1, 784)        # (60000, 784), pixels in [0, 1]
y_all = full.targets                                        # (60000,), labels 0 to 9
perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(0))
train_idx, val_idx = perm[:2000], perm[-5000:]
MEAN, STD = X_all[train_idx].mean(), X_all[train_idx].std()    # statistics of the training images only
X_train, y_train = (X_all[train_idx] - MEAN) / STD, y_all[train_idx]
X_val, y_val = (X_all[val_idx] - MEAN) / STD, y_all[val_idx]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1000)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=1000)
loss_fn = nn.CrossEntropyLoss()
print("train", tuple(X_train.shape), "validation", tuple(X_val.shape))


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n


def evaluate(model, loader, loss_fn):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            total += loss_fn(out, yb).item() * len(yb)
            correct += (out.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n


def new_history():
    return {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}


def record(history, model, optimizer):
    """Measure the model on the training and validation sets and append the results to the history."""
    tl, ta = evaluate(model, train_eval_loader, loss_fn)
    vl, va = evaluate(model, val_loader, loss_fn)
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl); history["val_acc"].append(va)
    history["lr"].append(optimizer.param_groups[0]["lr"])


def fit(model, optimizer, n_epochs, train_epoch_fn=train_one_epoch, scheduler=None, verbose=False):
    torch.manual_seed(1)                       # same order of the mini-batches for every run
    history = new_history()
    for epoch in range(n_epochs):
        train_epoch_fn(model, train_loader, loss_fn, optimizer)
        record(history, model, optimizer)
        if scheduler is not None:
            scheduler.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  train loss {history['train_loss'][-1]:.3f}  "
                  f"val loss {history['val_loss'][-1]:.3f}  val acc {history['val_acc'][-1]:.3f}")
    return history


def plot_history(history, title=""):
    e = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(e, history["train_loss"], label="train"); axes[0].plot(e, history["val_loss"], label="validation")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
    axes[1].plot(e, history["train_acc"], label="train"); axes[1].plot(e, history["val_acc"], label="validation")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
    fig.suptitle(title)
    plt.show()


def plot_compare(histories, key="val_loss"):
    plt.figure(figsize=(7, 3.5))
    for name, h in histories.items():
        plt.plot(range(1, len(h[key]) + 1), h[key], label=name)
    plt.xlabel("epoch"); plt.ylabel(key); plt.legend()
    plt.show()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Part A: provoke overfitting

### Exercise 1: a big network

Write `make_big_model()` that returns the network of the course:
`784 -> 512 -> 512 -> 10`, with a ReLU after each hidden layer.

- `nn.Linear(n_in, n_out)` is a fully connected layer (weights of shape `(n_out, n_in)` and a bias).
- `nn.ReLU()` applies $\max(0, z)$ element-wise.
- `nn.Sequential(layer1, layer2, ...)` chains layers: the output of one is the input of the next.

No softmax at the end: `nn.CrossEntropyLoss` takes the raw scores (logits).

> Hint: 5 modules in the `nn.Sequential`: Linear, ReLU, Linear, ReLU, Linear.

In [ ]:
def make_big_model():
    ### START CODE HERE ###
    model = None
    ### END CODE HERE ###
    return model


torch.manual_seed(0)
_m = make_big_model()
assert isinstance(_m, nn.Sequential), "return an nn.Sequential"
assert _m(torch.zeros(3, 784)).shape == (3, 10), "the output must have shape (batch, 10)"
assert count_parameters(_m) == 669_706, f"expected 669 706 parameters, found {count_parameters(_m)}"
print("Exercise validated! The network has", count_parameters(_m), "parameters for 2 000 images.")

### Exercise 2: train it

Create the model with `make_big_model()` and an optimiser **SGD with momentum**:
`torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)`. `model.parameters()` gives the optimiser all
the tensors it must update; `lr` is $\eta$; `momentum` is $\beta$ of the course.

Then train with `history_base = fit(model_base, optimizer, EPOCHS)`. Name the model `model_base`.

> Hint: three lines. `torch.manual_seed(0)` is already written so that everybody gets the same initial weights.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###
history_base = None
### END CODE HERE ###

assert history_base is not None and len(history_base["val_loss"]) == EPOCHS, "call fit(model_base, optimizer, EPOCHS)"
assert history_base["train_loss"][-1] < history_base["train_loss"][0], "the training loss should decrease"
plot_history(history_base, "Big network, 2 000 images")
print("Exercise validated!")

### Exercise 3: measure the overfitting

From `history_base`, compute:

- `gap_base`: final train accuracy minus final validation accuracy (the last element of a list `L` is `L[-1]`);
- `best_epoch_base`: the epoch (counted from 1) where the validation loss is the smallest.

> Hint: `np.argmin(list)` returns the index of the smallest element, counted from 0.

In [ ]:
### START CODE HERE ###
gap_base = None
best_epoch_base = None
### END CODE HERE ###

assert gap_base is not None and abs(gap_base - (history_base["train_acc"][-1] - history_base["val_acc"][-1])) < 1e-9
assert best_epoch_base == int(np.argmin(history_base["val_loss"])) + 1, "epochs are counted from 1"
assert gap_base > 0.08, f"the gap should be large (overfitting), found {gap_base:.3f}"
assert history_base["val_loss"][-1] > min(history_base["val_loss"]), "the validation loss should rise at the end"
print(f"Exercise validated! Gap {gap_base:.3f}, best validation loss at epoch {best_epoch_base} of {EPOCHS}.")

**What we see.** The training accuracy keeps climbing while the validation accuracy plateaus: the gap is
large. The validation loss reaches its minimum after a few epochs, then rises: the network becomes
over-confident on images it has never seen. Everything after the best epoch is wasted (even harmful) training.

# Part B: regularisation

### Exercise 4: the L2 penalty

Write `l2_penalty(model, lam)` that returns $\frac{\lambda}{2}\sum W^2$: the sum of the squares of all the
**weights** of the `nn.Linear` layers (not the biases), times `lam / 2`.

- `model.modules()` iterates over all the layers of the model; `isinstance(layer, nn.Linear)` tests the type.
- `layer.weight` is the weight tensor; `(layer.weight ** 2).sum()` is the sum of its squares.

> Hint: start from `total = 0.0` and add the sum of each Linear layer's squared weights.

In [ ]:
def l2_penalty(model, lam):
    ### START CODE HERE ###
    penalty = None
    ### END CODE HERE ###
    return penalty


_tiny = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))
with torch.no_grad():
    _tiny[0].weight.copy_(torch.tensor([[2.0, -1.0], [0.0, 1.0]])); _tiny[0].bias.fill_(5.0)
    _tiny[2].weight.copy_(torch.tensor([[1.0, 3.0]])); _tiny[2].bias.fill_(7.0)
_p = l2_penalty(_tiny, 0.5)
assert _p is not None, "return the penalty"
assert abs(float(_p) - 0.25 * (4 + 1 + 0 + 1 + 1 + 9)) < 1e-6, f"expected 4.0, found {float(_p)} (biases must not be counted)"
print("Exercise validated!")

### Exercise 5: one weight-decay update by hand

The gradient of the penalty is $\lambda w$, so the update is
$w \leftarrow w - \eta\left(\frac{\partial J}{\partial w} + \lambda w\right)$. Compute `w_new` for the numbers of
Exercise 1 of the course: $w = (2, -1)$, $\partial J/\partial w = (0.5, -0.5)$, $\eta = 0.1$, $\lambda = 0.5$.

The assert then checks your result against PyTorch: `torch.optim.SGD(..., weight_decay=0.5)` does exactly this.

In [ ]:
w = torch.tensor([2.0, -1.0])
grad = torch.tensor([0.5, -0.5])
lr, lam = 0.1, 0.5
### START CODE HERE ###
w_new = None
### END CODE HERE ###

_w = w.clone().requires_grad_(True)
_opt = torch.optim.SGD([_w], lr=lr, weight_decay=lam)
_w.grad = grad.clone()
_opt.step()
assert w_new is not None and torch.allclose(w_new, torch.tensor([1.85, -0.90])), f"expected (1.85, -0.90), found {w_new}"
assert torch.allclose(w_new, _w.detach()), "PyTorch's weight_decay should give the same result"
print("Exercise validated! Same result as the course and as torch.optim.SGD(weight_decay=0.5).")

### Exercise 6: train with weight decay

Same model, same optimiser as Exercise 2, plus `weight_decay=1e-2`. Store the history in `history_wd` and the
gap (final train accuracy minus final validation accuracy) in `gap_wd`. Name the model `model_wd`.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###
history_wd = None
gap_wd = None
### END CODE HERE ###

assert history_wd is not None and len(history_wd["val_loss"]) == EPOCHS
assert abs(gap_wd - (history_wd["train_acc"][-1] - history_wd["val_acc"][-1])) < 1e-9
assert optimizer.param_groups[0]["weight_decay"] == 1e-2, "use weight_decay=1e-2"
assert gap_wd < gap_base, f"weight decay should reduce the gap ({gap_wd:.3f} versus {gap_base:.3f})"
plot_compare({"no regularisation": history_base, "weight decay": history_wd}, "val_loss")
print(f"Exercise validated! Gap {gap_base:.3f} without, {gap_wd:.3f} with weight decay.")

### Exercise 7: inverted dropout by hand

Write `dropout_forward(a, p, mask)`: `mask` contains 1 (keep) and 0 (drop); return
$\tilde a = \dfrac{\text{mask} \odot a}{1 - p}$ (element-wise product, then the division).

The first assert uses Exercise 2 of the course: $a = (2, 1, 4, 3)$, $p = 0.5$, mask $(1, 0, 1, 0)$. The second
draws a random mask on a million values and checks that the mean is preserved, as we proved:
$\mathbb{E}[\tilde a_i] = a_i$.

> Hint: `torch.bernoulli(torch.full(shape, 1 - p))` draws a mask with $P(1) = 1 - p$ (used in the assert).

In [ ]:
def dropout_forward(a, p, mask):
    ### START CODE HERE ###
    out = None
    ### END CODE HERE ###
    return out


_out = dropout_forward(torch.tensor([2.0, 1.0, 4.0, 3.0]), 0.5, torch.tensor([1.0, 0.0, 1.0, 0.0]))
assert _out is not None and torch.allclose(_out, torch.tensor([4.0, 0.0, 8.0, 0.0])), f"expected (4, 0, 8, 0), found {_out}"
torch.manual_seed(0)
_a = torch.rand(1_000_000) * 3
_mask = torch.bernoulli(torch.full(_a.shape, 0.7))           # keep with probability 0.7, so p = 0.3
assert abs(dropout_forward(_a, 0.3, _mask).mean() - _a.mean()) < 0.01, "the mean must be preserved (divide by 1 - p)"
print("Exercise validated!")

### Exercise 8: a model with dropout

Write `make_dropout_model(p)`: the big network with an `nn.Dropout(p)` **after each hidden ReLU**
(Linear, ReLU, Dropout, Linear, ReLU, Dropout, Linear). `nn.Dropout(p)` drops each value with probability
`p` in training mode and does nothing in evaluation mode.

The asserts check the two modes: in `train()` mode two passes on the same input give different outputs (two
different random masks); in `eval()` mode they are identical.

In [ ]:
def make_dropout_model(p):
    ### START CODE HERE ###
    model = None
    ### END CODE HERE ###
    return model


_m = make_dropout_model(0.5)
assert _m is not None and sum(isinstance(l, nn.Dropout) for l in _m) == 2, "two nn.Dropout layers"
assert count_parameters(_m) == 669_706, "dropout adds no parameter"
_x = X_train[:5]
_m.train()
assert not torch.allclose(_m(_x), _m(_x)), "in train mode two passes should differ"
_m.eval()
assert torch.allclose(_m(_x), _m(_x)), "in eval mode two passes should be identical"
print("Exercise validated!")

### Exercise 9: train with dropout

Train `model_drop = make_dropout_model(0.5)` with the optimiser of Exercise 2 (no weight decay). Store
`history_drop` and `gap_drop`.

`fit` calls `model.train()` before each epoch (inside `train_one_epoch`) and `model.eval()` to measure
(inside `evaluate`): the train accuracy is measured without dropout, like the validation accuracy.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###
history_drop = None
gap_drop = None
### END CODE HERE ###

assert history_drop is not None and len(history_drop["val_loss"]) == EPOCHS
assert abs(gap_drop - (history_drop["train_acc"][-1] - history_drop["val_acc"][-1])) < 1e-9
assert gap_drop < gap_base, f"dropout should reduce the gap ({gap_drop:.3f} versus {gap_base:.3f})"
plot_compare({"no regularisation": history_base, "weight decay": history_wd, "dropout 0.5": history_drop}, "val_loss")
print(f"Exercise validated! Gap {gap_base:.3f} without, {gap_drop:.3f} with dropout.")

### Exercise 10: early stopping with the best checkpoint

Complete `fit_early_stopping(model, optimizer, max_epochs, patience)`. After each epoch the validation loss
`val_loss` is known. Then:

- if `val_loss < best_loss`: it is the best epoch so far. Save `best_loss`, `best_epoch = epoch`, reset
  `wait = 0` and save a copy of the weights: `best_state = copy.deepcopy(model.state_dict())`.
  (`state_dict()` returns references to the live tensors, which keep changing: without `deepcopy` your "copy"
  would follow the training.)
- otherwise: `wait += 1`, and `break` out of the loop if `wait >= patience`.

After the loop, reload the best weights with `model.load_state_dict(best_state)`.

In [ ]:
def fit_early_stopping(model, optimizer, max_epochs, patience):
    torch.manual_seed(1)
    history = new_history()
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, max_epochs + 1):
        train_one_epoch(model, train_loader, loss_fn, optimizer)
        record(history, model, optimizer)
        val_loss = history["val_loss"][-1]
        ### START CODE HERE ###

        ### END CODE HERE ###
    ### START CODE HERE ###

    ### END CODE HERE ###
    return history, best_epoch


torch.manual_seed(0)
model_es = make_big_model()
optimizer = torch.optim.SGD(model_es.parameters(), lr=0.05, momentum=0.9)
history_es, best_epoch_es = fit_early_stopping(model_es, optimizer, max_epochs=EPOCHS, patience=5)

assert best_epoch_es == int(np.argmin(history_es["val_loss"])) + 1, "best_epoch must be the epoch of the smallest validation loss"
assert len(history_es["val_loss"]) <= best_epoch_es + 5, "training must stop `patience` epochs after the best one"
_vl, _va = evaluate(model_es, val_loader, loss_fn)
assert abs(_vl - min(history_es["val_loss"])) < 1e-5, "the model must be reloaded with the best weights"
print(f"Exercise validated! Stopped after {len(history_es['val_loss'])} epochs, best epoch {best_epoch_es}, "
      f"validation accuracy {_va:.3f}.")

### Exercise 11: the comparison table

Build a pandas DataFrame `results` with one row per run (index: `"baseline"`, `"weight decay"`, `"dropout"`,
`"early stopping"`) and 3 columns:

- `"val_acc"`: the validation accuracy of the **final** model (for early stopping: the reloaded best model,
  that is `history_es["val_acc"][best_epoch_es - 1]`);
- `"gap"`: train accuracy minus validation accuracy of that same model;
- `"best_val_loss"`: the smallest validation loss of the run.

A small helper `row(history, i)` is a good idea: it returns the 3 values for epoch index `i` (`-1` for the last).

> Hint: `pd.DataFrame.from_dict({"baseline": [a, b, c], ...}, orient="index", columns=["val_acc", "gap", "best_val_loss"])`.

In [ ]:
### START CODE HERE ###
results = None
### END CODE HERE ###

assert results is not None and results.shape == (4, 3), "4 rows, 3 columns"
assert list(results.index) == ["baseline", "weight decay", "dropout", "early stopping"]
assert abs(results.loc["dropout", "gap"] - gap_drop) < 1e-9
assert abs(results.loc["early stopping", "val_acc"] - _va) < 1e-9, "early stopping: use the reloaded best epoch"
assert results.loc["baseline", "gap"] == results["gap"].max(), "the baseline should have the largest gap"
print(results.round(3))
print("Exercise validated!")

**What we see.** Every tool reduces the gap between train and validation. Weight decay and dropout make the
training harder (lower train accuracy) so that the network cannot memorise; early stopping simply keeps the
epoch where the validation loss was the best, for free. On 2 000 images no tool works miracles on the
validation accuracy: the real cure for overfitting remains more data.

# Part C: batch normalisation

### Exercise 12: batch norm forward by hand

Write `batchnorm_forward(Z, gamma, beta, eps=1e-5)` for a batch `Z` of shape `(m, n)` (m examples, n neurons).
For **each neuron** (each column):

$$\mu_B = \frac{1}{m}\sum_i z^{(i)}, \quad \sigma_B^2 = \frac{1}{m}\sum_i (z^{(i)} - \mu_B)^2, \quad
\hat z = \frac{z - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \quad y = \gamma \hat z + \beta$$

- `Z.mean(dim=0)` averages over the examples (shape `(n,)`).
- `Z.var(dim=0, unbiased=False)` is the variance with $\frac{1}{m}$ (the default `unbiased=True` divides by $m - 1$).
- `torch.sqrt`; broadcasting does the rest: `(Z - mu)` has shape `(m, n)`.

The asserts compare with `nn.BatchNorm1d` in training mode, and with Exercise 3 of the course.

In [ ]:
def batchnorm_forward(Z, gamma, beta, eps=1e-5):
    ### START CODE HERE ###
    Y = None
    ### END CODE HERE ###
    return Y


_y = batchnorm_forward(torch.tensor([[1.0], [3.0], [3.0], [9.0]]), torch.tensor([2.0]), torch.tensor([0.5]))
assert _y is not None and torch.allclose(_y.flatten(), torch.tensor([-1.5, -1 / 6, -1 / 6, 23 / 6]), atol=1e-4), \
    f"course Exercise 3 expects (-1.5, -0.167, -0.167, 3.833), found {_y.flatten()}"
torch.manual_seed(0)
_Z = torch.randn(8, 3) * 4 + 2
_bn = nn.BatchNorm1d(3)
with torch.no_grad():
    _bn.weight.copy_(torch.tensor([1.0, 2.0, 0.5])); _bn.bias.copy_(torch.tensor([0.0, -1.0, 3.0]))
_bn.train()
assert torch.allclose(batchnorm_forward(_Z, _bn.weight.detach(), _bn.bias.detach()), _bn(_Z).detach(), atol=1e-5), \
    "your result differs from nn.BatchNorm1d: normalise each column, with the biased variance"
print("Exercise validated!")

### Exercise 13: the running statistics

A fresh `nn.BatchNorm1d(3)` starts with `running_mean = 0` and `running_var = 1`. After **one** forward pass in
training mode on the batch `_Z` above, PyTorch updates them with `momentum = 0.1`:

$$\mu_{\text{run}} \leftarrow 0.9\,\mu_{\text{run}} + 0.1\,\mu_B, \qquad
\sigma^2_{\text{run}} \leftarrow 0.9\,\sigma^2_{\text{run}} + 0.1\,\sigma^2_{B,\text{unbiased}}$$

(careful: for the running variance PyTorch uses the **unbiased** variance, divided by $m - 1$).
Compute `expected_mean` and `expected_var` (tensors of shape `(3,)`) from `_Z`.

In [ ]:
### START CODE HERE ###
expected_mean = None
expected_var = None
### END CODE HERE ###

_bn2 = nn.BatchNorm1d(3)
_bn2.train()
_ = _bn2(_Z)
assert expected_mean is not None and torch.allclose(expected_mean, _bn2.running_mean, atol=1e-5), "running mean differs"
assert expected_var is not None and torch.allclose(expected_var, _bn2.running_var, atol=1e-4), "running var differs (unbiased variance?)"
print("Exercise validated! In eval() mode, the layer normalises with these running statistics.")

### Exercise 14: batch norm speeds up training

Write `make_bn_model()`: the big network with an `nn.BatchNorm1d(512)` between each hidden `Linear` and its
`ReLU` (Linear, BatchNorm1d, ReLU, Linear, BatchNorm1d, ReLU, Linear).

Then train `make_big_model()` and `make_bn_model()` with **plain SGD** (`lr=0.01`, no momentum) for
`EPOCHS_SHORT` epochs. Store `history_plain` and `history_bn`.

In [ ]:
def make_bn_model():
    ### START CODE HERE ###
    model = None
    ### END CODE HERE ###
    return model


### START CODE HERE ###
history_plain, history_bn = None, None
### END CODE HERE ###

assert make_bn_model() is not None and sum(isinstance(l, nn.BatchNorm1d) for l in make_bn_model()) == 2
assert history_bn is not None and len(history_bn["train_loss"]) == EPOCHS_SHORT
assert history_bn["train_loss"][-1] < 0.8 * history_plain["train_loss"][-1], "batch norm should train clearly faster"
plot_compare({"plain SGD": history_plain, "plain SGD + batch norm": history_bn}, "train_loss")
print(f"Exercise validated! Final train loss {history_plain['train_loss'][-1]:.3f} without, "
      f"{history_bn['train_loss'][-1]:.3f} with batch norm.")

**What we see.** With the same optimiser and the same learning rate, the network with batch norm goes down
much faster: each layer receives inputs of mean 0 and variance 1, whatever the layers before it do, so the
same step size suits every layer.

# Part D: optimisers

### Exercise 15: one momentum step

Write `momentum_step(w, v, g, lr, beta)` that returns the new `(w, v)` with PyTorch's convention:

$$v \leftarrow \beta v + g, \qquad w \leftarrow w - \eta\, v$$

The asserts replay Exercise 4 of the course ($J = w^2$, $g = 2w$, $w = 1$, $\eta = 0.1$, $\beta = 0.9$: we
expect 0.8 then 0.46) and compare 3 steps with `torch.optim.SGD(momentum=0.9)` on a random tensor.

In [ ]:
def momentum_step(w, v, g, lr, beta):
    ### START CODE HERE ###

    ### END CODE HERE ###
    return w, v


_w, _v = 1.0, 0.0
_ws = []
for _ in range(2):
    _w, _v = momentum_step(_w, _v, 2 * _w, 0.1, 0.9)
    _ws.append(_w)
assert np.allclose(_ws, [0.8, 0.46]), f"course Exercise 4 expects 0.8 then 0.46, found {_ws}"
torch.manual_seed(0)
_p = torch.randn(4, requires_grad=True)
_opt = torch.optim.SGD([_p], lr=0.05, momentum=0.9)
_w, _v = _p.detach().clone(), torch.zeros(4)
for _ in range(3):
    _opt.zero_grad(); (_p ** 4).sum().backward(); _g = _p.grad.detach().clone(); _opt.step()
    _w, _v = momentum_step(_w, _v, _g, 0.05, 0.9)
assert torch.allclose(_w, _p.detach(), atol=1e-6), "3 steps should match torch.optim.SGD(momentum=0.9)"
print("Exercise validated!")

### Exercise 16: one Adam step

Write `adam_step(w, m, s, g, t, lr, beta1=0.9, beta2=0.999, eps=1e-8)` that returns the new `(w, m, s)`.
`t` is the step number, starting at 1.

$$m \leftarrow \beta_1 m + (1-\beta_1) g, \quad s \leftarrow \beta_2 s + (1-\beta_2) g^2, \quad
\hat m = \frac{m}{1-\beta_1^t}, \quad \hat s = \frac{s}{1-\beta_2^t}, \quad w \leftarrow w - \eta \frac{\hat m}{\sqrt{\hat s} + \varepsilon}$$

Use `torch.sqrt` (the inputs are tensors). The asserts check that the first step has size $\eta$ (course
Exercise 4) and compare 3 steps with `torch.optim.Adam`.

In [ ]:
def adam_step(w, m, s, g, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    ### START CODE HERE ###

    ### END CODE HERE ###
    return w, m, s


_w1, _, _ = adam_step(torch.tensor([1.0]), torch.zeros(1), torch.zeros(1), torch.tensor([2.0]), 1, 0.1)
assert torch.allclose(_w1, torch.tensor([0.9])), f"first Adam step from w = 1 with lr 0.1 should give 0.9, found {_w1}"
torch.manual_seed(0)
_p = torch.randn(4, requires_grad=True)
_opt = torch.optim.Adam([_p], lr=0.01)
_w, _m, _s = _p.detach().clone(), torch.zeros(4), torch.zeros(4)
for _t in range(1, 4):
    _opt.zero_grad(); (_p ** 4).sum().backward(); _g = _p.grad.detach().clone(); _opt.step()
    _w, _m, _s = adam_step(_w, _m, _s, _g, _t, 0.01)
assert torch.allclose(_w, _p.detach(), atol=1e-6), "3 steps should match torch.optim.Adam"
print("Exercise validated!")

### Exercise 17: SGD versus momentum versus Adam

Train the big network three times for `EPOCHS_SHORT` epochs, with the same initial weights
(`torch.manual_seed(0)` before each `make_big_model()`):

- `"SGD"`: `torch.optim.SGD(params, lr=0.05)`
- `"momentum"`: `torch.optim.SGD(params, lr=0.05, momentum=0.9)`
- `"Adam"`: `torch.optim.Adam(params, lr=1e-3)` (the usual default learning rate of Adam)

Store the three histories in a dictionary `histories_opt` with these keys.

> Hint: a dictionary of functions `{"SGD": lambda p: torch.optim.SGD(p, lr=0.05), ...}` and a loop.

In [ ]:
### START CODE HERE ###
histories_opt = {}
### END CODE HERE ###

assert set(histories_opt) == {"SGD", "momentum", "Adam"}, "three keys: SGD, momentum, Adam"
_final = {k: h["train_loss"][-1] for k, h in histories_opt.items()}
assert _final["momentum"] < _final["SGD"] and _final["Adam"] < _final["SGD"], f"momentum and Adam should beat plain SGD: {_final}"
plot_compare(histories_opt, "train_loss")
print("Exercise validated!", {k: round(v, 3) for k, v in _final.items()})

### Exercise 18: a cosine learning-rate schedule

`torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n)` decreases the learning rate of `optimizer`
from its initial value to 0 along half a cosine over `n` calls to `scheduler.step()`. `fit` accepts a
`scheduler` argument and calls `scheduler.step()` after each epoch; the history records the learning rate
used at each epoch in `"lr"`.

Create a model, an SGD optimiser with `lr=0.05, momentum=0.9`, a cosine scheduler with `T_max=EPOCHS_SHORT`, and
train: `history_cos = fit(model, optimizer, EPOCHS_SHORT, scheduler=scheduler)`.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###
history_cos = None
### END CODE HERE ###

assert history_cos is not None and len(history_cos["lr"]) == EPOCHS_SHORT
assert abs(history_cos["lr"][0] - 0.05) < 1e-9, "the first epoch uses the initial learning rate"
assert all(a > b for a, b in zip(history_cos["lr"], history_cos["lr"][1:])), "the learning rate must decrease"
assert history_cos["lr"][-1] < 0.01, "at the last epoch the learning rate should be small"
plt.figure(figsize=(5, 3)); plt.plot(range(1, EPOCHS_SHORT + 1), history_cos["lr"], "o-"); plt.xlabel("epoch"); plt.ylabel("lr"); plt.show()
print(f"Exercise validated! Final train loss {history_cos['train_loss'][-1]:.3f} with the schedule, "
      f"{histories_opt['momentum']['train_loss'][-1]:.3f} without.")

**What we see.** Momentum and Adam reach a lower training loss than plain SGD in the same number of epochs.
The schedule takes large steps at the start and small steps at the end: the last epochs settle the weights
instead of bouncing around, which usually gives a lower and smoother final loss.

# Part E: the training doctor

Three colleagues send you their sick training runs. For each one: run the cell, read the numbers, write your
**diagnosis** in the markdown cell (what is wrong, how you know), then fix it in the code cell. Use the
diagnosis table of the course.

### Patient 1

In [ ]:
torch.manual_seed(0)
model = make_big_model()
optimizer = torch.optim.SGD(model.parameters(), lr=1.0, momentum=0.9)
history_p1 = fit(model, optimizer, 3, verbose=True)

### Exercise 19: diagnose and cure patient 1

Your diagnosis:

> *Your answer here.*

Now cure it: same model, same optimiser type, a learning rate that works. Store the history in `history_fix1`
(`EPOCHS_SHORT` epochs).

In [ ]:
torch.manual_seed(0)
model = make_big_model()
### START CODE HERE ###
history_fix1 = None
### END CODE HERE ###

assert history_fix1 is not None and all(math.isfinite(v) for v in history_fix1["train_loss"]), "the loss must stay finite"
assert history_fix1["val_acc"][-1] > 0.7, f"validation accuracy should exceed 0.7, found {history_fix1['val_acc'][-1]:.3f}"
print("Exercise validated! Patient 1 cured:", round(history_fix1["val_acc"][-1], 3))

### Patient 2

This colleague wrote their own training epoch. The learning rate is the one that works.

In [ ]:
def buggy_train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()


torch.manual_seed(0)
model = make_big_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
history_p2 = fit(model, optimizer, 3, train_epoch_fn=buggy_train_one_epoch, verbose=True)

### Exercise 20: diagnose and cure patient 2

Your diagnosis:

> *Your answer here.*

Write `fixed_train_one_epoch` (the 5 lines of the training loop inside the batch loop), then train a fresh model
with it for `EPOCHS_SHORT` epochs: `history_fix2 = fit(model, optimizer, EPOCHS_SHORT, train_epoch_fn=fixed_train_one_epoch)`.

In [ ]:
def fixed_train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    for xb, yb in loader:
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###


torch.manual_seed(0)
model = make_big_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
### START CODE HERE ###
history_fix2 = None
### END CODE HERE ###

assert history_fix2 is not None and all(math.isfinite(v) for v in history_fix2["train_loss"]), "the loss must stay finite"
assert history_fix2["val_acc"][-1] > 0.7, f"validation accuracy should exceed 0.7, found {history_fix2['val_acc'][-1]:.3f}"
print("Exercise validated! Patient 2 cured:", round(history_fix2["val_acc"][-1], 3))

### Patient 3

This colleague wanted a deep network: 8 hidden layers of 128 sigmoid neurons.

In [ ]:
def make_deep_sigmoid():
    layers, n_in = [], 784
    for _ in range(8):
        layers += [nn.Linear(n_in, 128), nn.Sigmoid()]
        n_in = 128
    return nn.Sequential(*layers, nn.Linear(n_in, 10))


torch.manual_seed(0)
model = make_deep_sigmoid()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
history_p3 = fit(model, optimizer, 3, verbose=True)
print("ln(10) =", round(math.log(10), 3))

### Exercise 21: diagnose and cure patient 3

Your diagnosis:

> *Your answer here.*

Write `make_deep_fixed()`: the same 8 hidden layers of 128 neurons but with `nn.ReLU()`, and He initialisation
of every `nn.Linear`: `nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")` and `nn.init.zeros_(layer.bias)`
(the trailing underscore means "in place"). Then train it with the same optimiser for `EPOCHS_SHORT` epochs:
`history_fix3`.

> Hint: build the `nn.Sequential` as in `make_deep_sigmoid`, then loop over `model.modules()` to initialise.

In [ ]:
def make_deep_fixed():
    ### START CODE HERE ###
    model = None
    ### END CODE HERE ###
    return model


torch.manual_seed(0)
model = make_deep_fixed()
assert model is not None and sum(isinstance(l, nn.Linear) for l in model) == 9, "8 hidden layers + 1 output layer"
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
### START CODE HERE ###
history_fix3 = None
### END CODE HERE ###

assert history_fix3 is not None and history_fix3["train_loss"][-1] < 1.2, "the loss should leave ln(10) = 2.30"
assert history_fix3["val_acc"][-1] > 0.55, f"validation accuracy should exceed 0.55, found {history_fix3['val_acc'][-1]:.3f}"
print("Exercise validated! Patient 3 cured:", round(history_fix3["val_acc"][-1], 3))

### Exercise 22: gradient clipping

After `loss.backward()`, compute by hand the norm of the **whole** gradient (all parameters together):

$$\lVert g \rVert = \sqrt{\sum_{\text{param}} \sum_{\text{entries}} g^2}$$

Store it in `total_norm` (a Python float). Then call
`torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`: it returns the norm before clipping and
rescales the gradients in place if the norm exceeds 1.

> Hint: `sum((p.grad ** 2).sum() for p in model.parameters())`, then `math.sqrt(float(...))`.

In [ ]:
torch.manual_seed(0)
model = make_big_model()
loss = loss_fn(model(X_train[:256]), y_train[:256]) * 50      # a loss scaled up on purpose: large gradients
loss.backward()
### START CODE HERE ###
total_norm = None
### END CODE HERE ###
returned = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
new_norm = math.sqrt(float(sum((p.grad ** 2).sum() for p in model.parameters())))

assert total_norm is not None and abs(total_norm - float(returned)) < 1e-3 * total_norm, "your norm differs from PyTorch's"
assert total_norm > 1.0, "the gradient norm should be larger than 1 here"
assert abs(new_norm - 1.0) < 1e-3, "after clipping, the norm should be exactly max_norm = 1"
print(f"Exercise validated! Norm {total_norm:.2f} before clipping, {new_norm:.3f} after.")

# Going further

### Exercise 23 (bonus): your best model

Combine what you learned: batch norm, dropout, AdamW (`torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2)`),
a cosine schedule, early stopping... Build `model_best` and train it on the same 2 000 images so that the
validation accuracy of the final model, `best_val_acc`, beats the final validation accuracy of the baseline
(Exercise 2) by at least 0.005.

> Hint: `fit_early_stopping` does not take a scheduler; you may write your own loop or simply use `fit` with a
> scheduler and a reasonable number of epochs.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###
best_val_acc = None
### END CODE HERE ###

assert best_val_acc is not None and abs(best_val_acc - evaluate(model_best, val_loader, loss_fn)[1]) < 1e-9, \
    "best_val_acc must be the validation accuracy of the final model_best"
assert best_val_acc > history_base["val_acc"][-1] + 0.005, \
    f"{best_val_acc:.3f} does not beat the final baseline ({history_base['val_acc'][-1]:.3f}) by 0.005"
print(f"Exercise validated! {best_val_acc:.3f} versus {history_base['val_acc'][-1]:.3f} for the final baseline.")